# ViHSD Mixture of Experts experiment

This notebook keeps the same Colab setup as before, but it is now designed for running multiple experiment versions in one session. Instead of modifying `configs/vihsd.yaml` manually, you define a list of experiment variants, each with its own run ID, overrides, and training profile. All runs follow the same training and evaluation workflow, which makes later comparison fair and easy.

Use this notebook when you want to compare:
- baseline vs stronger MoE
- different expert counts or top-k values
- different learning rates and losses
- smoke tests vs full runs

The repository code remains the source of truth, and each run still saves its own `resolved_config.yaml`, metrics, and predictions.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('Hugging Face and W&B authentication configured from Colab Secrets.')

## 4. Define experiment versions

This notebook now supports running multiple versions in one pass. Each item in `EXPERIMENTS` is a separate experiment configuration, and each run still uses the same training and evaluation pipeline.

### How to use this section
- Each experiment is a dictionary.
- `run_id` is the unique folder name for that run.
- `overrides` contains YAML-style keys using the same dotted format as before.
- `smoke_test` turns on the quick training profile for debugging.
- `description` is only for readability in this notebook.

### Rules
- Keep the repository YAML as the shared baseline.
- Only change values in `overrides` when testing variant settings.
- Use a unique `run_id` for each experiment so the saved results remain separate.
- Use `model.architecture` to switch between architecture variants such as `current_moe` and `stronger_moe`.
- Omit keys you do not want to override, and the default YAML value is used.

### Example structure
```python
EXPERIMENTS = [
    {
        'name': 'baseline-current-moe',
        'description': 'Current baseline architecture',
        'run_id': 'baseline-current-moe',
        'smoke_test': False,
        'overrides': {
            'model.architecture': 'current_moe',
            'training.learning_rate': 2e-4,
            'training.epochs': 5,
        },
    },
    {
        'name': 'stronger-moe',
        'description': 'Stronger MoE candidate',
        'run_id': 'stronger-moe-v1',
        'smoke_test': False,
        'overrides': {
            'model.architecture': 'stronger_moe',
            'model.num_experts': 8,
            'model.top_k': 2,
            'training.loss_type': 'focal',
        },
    },
]
```

This is the recommended way to compare multiple versions in one Colab session without editing the YAML file.

In [ ]:
# Recommended approach: define multiple experiment versions in one list.
# Each entry will be trained in sequence with the same workflow.
EXPERIMENTS = [
    {
        'name': 'baseline-current-moe',
        'description': 'Current baseline architecture',
        'run_id': 'baseline-current-moe',
        'smoke_test': False,
        'overrides': {
            'model.architecture': 'current_moe',
            'training.learning_rate': 2e-4,
            'training.epochs': 5,
        },
    },
    {
        'name': 'stronger-moe',
        'description': 'Stronger MoE candidate',
        'run_id': 'stronger-moe-v1',
        'smoke_test': False,
        'overrides': {
            'model.architecture': 'stronger_moe',
            'model.num_experts': 8,
            'model.top_k': 2,
            'training.loss_type': 'focal',
            'training.focal_gamma': 2.0,
        },
    },
    {
        'name': 'stronger-moe-smoke',
        'description': 'Quick smoke test for debugging',
        'run_id': 'smoke-stronger-moe',
        'smoke_test': True,
        'overrides': {
            'model.architecture': 'stronger_moe',
            'model.num_experts': 8,
            'model.top_k': 2,
        },
    },
]

# If you want to run just one experiment instead, replace EXPERIMENTS with a single dict.
# Example:
# EXPERIMENTS = [{
#     'name': 'single-baseline',
#     'run_id': 'single-baseline',
#     'smoke_test': False,
#     'overrides': {'model.architecture': 'current_moe'},
# }]

print(f'Prepared {len(EXPERIMENTS)} experiment version(s).')
for exp in EXPERIMENTS:
    print(f"- {exp['name']}: {exp['run_id']}")

## 5. Train all configured experiment versions

This step runs the full training workflow for every item in `EXPERIMENTS`. The code below keeps the same repo workflow, but loops over multiple versions in sequence.

What happens in this cell:
1. each experiment gets converted into a `train.py` command
2. the command is executed in the same Colab kernel
3. a unique run folder is created for each version
4. the resolved config and training history are saved for each run
5. the best checkpoint is automatically selected per version

This is the recommended pattern when you want a fair comparison across multiple architectures or hyperparameter sets.

In [ ]:
import json
import sys
from train import main as train_main


def run_training_for_experiment(exp):
    cmd = ['--config', 'configs/vihsd.yaml']
    cmd.append('--smoke-test' if exp.get('smoke_test', False) else '--no-smoke-test')
    run_id = exp.get('run_id')
    if run_id:
        cmd.extend(['--run-id', run_id])
    for key, value in exp.get('overrides', {}).items():
        cmd.extend(['--set', f'{key}={json.dumps(value)}'])

    print('\n' + '=' * 80)
    print(f"Training experiment: {exp.get('name', run_id)}")
    print('Command: python train.py', ' '.join(cmd))
    print('=' * 80)

    previous_argv = sys.argv
    try:
        sys.argv = ['train.py', *cmd]
        train_main()
    finally:
        sys.argv = previous_argv


for exp in EXPERIMENTS:
    run_training_for_experiment(exp)

print('\nAll configured experiments finished.')

## 6. Evaluate each experiment run

After training, evaluate every saved run using its run ID so that the test metrics and saved predictions remain tied to the exact run configuration.

This makes the comparison fair because each version is evaluated using its own `resolved_config.yaml` and best checkpoint.

In [ ]:
import subprocess

for exp in EXPERIMENTS:
    run_id = exp.get('run_id')
    if not run_id:
        continue
    cmd = ['python', 'evaluate.py', '--config', 'configs/vihsd.yaml', '--run-id', run_id]
    print('\n' + '-' * 80)
    print(f"Evaluating: {exp.get('name', run_id)}")
    print('Command:', ' '.join(cmd))
    print('-' * 80)
    subprocess.run(cmd, check=True)

print('\nAll evaluations finished.')


## 7. Show evaluation summary

This final section reads the saved run metrics from each experiment result folder and displays the evaluation summary in a compact table inside Colab. It is useful when you want to compare variants side-by-side without opening multiple JSON files.


In [ ]:
import json
from pathlib import Path


def load_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return None


summary_rows = []

for exp in EXPERIMENTS:
    run_id = exp.get('run_id')
    if not run_id:
        continue

    metrics_path = Path('results') / run_id / 'run_metrics.json'
    metrics = load_json(metrics_path)
    if metrics is None:
        print(f"No evaluation file found for {run_id}")
        continue

    test = metrics.get('test', {})
    summary_rows.append({
        'experiment': exp.get('name', run_id),
        'run_id': run_id,
        'accuracy': test.get('accuracy'),
        'macro_f1': test.get('macro_f1'),
        'weighted_f1': test.get('weighted_f1'),
        'loss': test.get('loss'),
    })

if summary_rows:
    print('\nEvaluation summary table')
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    display(df.sort_values(by=['macro_f1', 'weighted_f1'], ascending=False))
else:
    print('No completed experiments with saved metrics were found.')
